In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [4]:
# =============================================================================
# 1. DATA LOADING AND PREP
# =============================================================================
all_prices = pd.read_excel('All_Raw_Data.xlsx', index_col=0)
all_returns = all_prices.pct_change().dropna() * 100

# NEW: Load LSEG forecasted alphas (ticker in col0, alpha in col1, no headers).
# Assumes annual % alphas; match on ticker names.
lseg_forecast_raw = pd.read_excel('LSEG_Pred_Alpha.xlsx', header=None)

if lseg_forecast_raw.shape[1] == 2:
	# Already in [ticker, alpha] columns
	lseg_forecast = lseg_forecast_raw.copy()
	lseg_forecast.columns = ['ticker', 'alpha_forecast_annual_pct']
elif lseg_forecast_raw.shape[0] >= 2:
	# In 2xN layout: first row tickers, second row alphas
	lseg_forecast = lseg_forecast_raw.iloc[:2].T
	lseg_forecast.columns = ['ticker', 'alpha_forecast_annual_pct']
else:
	raise ValueError("Unexpected format in LSEG_Pred_Alpha.xlsx")

lseg_forecast['alpha_forecast_annual_pct'] = pd.to_numeric(
	lseg_forecast['alpha_forecast_annual_pct'], errors='coerce'
)
alpha_forecast_dict = dict(zip(lseg_forecast['ticker'], lseg_forecast['alpha_forecast_annual_pct']))

In [5]:
# =============================================================================
# 2. KEY INPUT PARAMETERS
# =============================================================================
rf_monthly_pct = 0.37 / 100
benchmark_col = 'ACWI.O'
securities = [c for c in all_returns.columns if c != benchmark_col]
rf_annual_pct = (1 + rf_monthly_pct)**12 - 1
portfolio_value = 50000000

In [7]:
# =============================================================================
# 3. CAPM REGRESSIONS: BETA & SIGMA_E (History Only)
# Regressions estimate structural params (beta, sigma_e), NOT alphas.
# Alphas from history are noisy; overwritten by LSEG forecasts below.
# =============================================================================
results = {}
benchmark_ret = all_returns[benchmark_col]
for ticker in securities:
    asset_ret = all_returns[ticker]
    valid = ~(asset_ret.isna() | benchmark_ret.isna())
    if valid.sum() < 12:
        results[ticker] = {'beta': 1.0, 'sigmae_monthly_pct': 1.0}
        continue
    
    y = asset_ret[valid] - rf_monthly_pct * 100
    x = benchmark_ret[valid] - rf_monthly_pct * 100
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    
    # Store beta & sigma_e; ignore regression alpha.
    beta = float(model.params.iloc[1])
    sigmae_monthly = np.sqrt(float(model.mse_resid))
    results[ticker] = {'beta': beta, 'sigmae_monthly_pct': sigmae_monthly}

In [8]:
# =============================================================================
# 4. ANNUALISATION + ADD FORECASTED ALPHAS
# sigma_e: sqrt(12). Forecast alphas used directly (annual % assumed).
# =============================================================================
for t in results:
    r = results[t]
    r['sigmae_annual_pct'] = r['sigmae_monthly_pct'] * np.sqrt(12)
    # LSEG forecast: overwrite (key TB step!).
    r['alpha_forecast_annual_pct'] = alpha_forecast_dict.get(t, 0.0)  # Default 0 if missing.

# Market stats.
ER_M_monthly = all_returns[benchmark_col].mean()
sigma_M_monthly = all_returns[benchmark_col].std()
ER_M_annual_pct = (1 + ER_M_monthly/100)**12 - 1
sigma_M_annual_pct = sigma_M_monthly * np.sqrt(12)
var_M_annual = (sigma_M_annual_pct / 100)**2
RPM_annual = ER_M_annual_pct - rf_annual_pct

print("Top 5 forecasted alphas (LSEG)")
for t in sorted(results, key=lambda x: results[x]['alpha_forecast_annual_pct'], reverse=True)[:5]:
    r = results[t]
    print(f"{t:15} f_alpha {r['alpha_forecast_annual_pct']:.2f}, sigmae_ann {r['sigmae_annual_pct']:.2f}")


Top 5 forecasted alphas (LSEG)
000660.KS       f_alpha 0.34, sigmae_ann 56.54
BBRM.NS         f_alpha 0.29, sigmae_ann 41.69
MSFT.O          f_alpha 0.16, sigmae_ann 22.48
FUTU.OQ         f_alpha 0.14, sigmae_ann 73.19
6723.T          f_alpha 0.14, sigmae_ann 44.30


In [10]:
# =============================================================================
# 5. ACTIVE PORTFOLIO (Eq 8.21-22): Use Forecast Alphas!
# w*_i = alpha_forecast_i / sigma^2_ei (historical variance).
# Positive/negative forecasts → long/short active bets.
# =============================================================================
w0i = {}
for t in securities:
    alpha_f = results[t]['alpha_forecast_annual_pct']
    w0i[t] = (alpha_f / 100) / (results[t]['sigmae_annual_pct'] / 100)**2

sum_w0i_abs = sum(abs(v) for v in w0i.values())
wAi = {t: w0i[t] / sum_w0i_abs for t in securities} if sum_w0i_abs > 0 else {t: 1/len(securities) for t in securities}

print("\nActive PF weights sum=100% (using LSEG forecasts, Eq 8.22)")
for t in sorted(wAi, key=lambda k: abs(wAi[k]), reverse=True)[:10]:
    print(f"{t:15} {wAi[t]*100:.1f}")


Active PF weights sum=100% (using LSEG forecasts, Eq 8.22)
IEF.O           20.6
VTIP.O          18.2
SDEU.L          15.0
5108.T          7.7
MSFT.O          6.9
IBGL.L          5.0
BLK.N           5.0
SASY.PA         4.3
BBRM.NS         3.6
AM.PA           2.5


In [11]:
# =============================================================================
# 6. ACTIVE AGGREGATES (Eq 8.23)
# alpha_A, beta_A from forecasts/history mix. sigma_eA diversified.
# =============================================================================
alpha_A_dec = sum(wAi[t] * (results[t]['alpha_forecast_annual_pct']/100) for t in securities)
sigmae_A_dec = np.sqrt(sum((wAi[t]**2 * (results[t]['sigmae_annual_pct']/100)**2) for t in securities))
beta_A = sum(wAi[t] * results[t]['beta'] for t in securities)
alpha_A_pct = alpha_A_dec * 100
sigmae_A_pct = sigmae_A_dec * 100

print(f"\nActive PF (forecast-driven): alpha_A {alpha_A_pct:.2f}, sigma_eA {sigmae_A_pct:.2f}, beta_A {beta_A:.2f}")


Active PF (forecast-driven): alpha_A 0.08, sigma_eA 4.12, beta_A 0.02


In [12]:
# =============================================================================
# 7. OPTIMAL MIX (Eq 8.24-25)
# w0: raw active weight. wA: beta-neutralises overall portfolio.
# =============================================================================
w0 = (alpha_A_dec / sigmae_A_dec**2) / ((RPM_annual / 100) / var_M_annual) if sigmae_A_dec > 0 else 0
wA = w0 / (1 + (1 - beta_A) * w0) if abs(1 + (1 - beta_A) * w0) > 1e-6 else 0
wM = 1 - wA

print(f"\nTB Optimal Mix (Eq 8.24-25, forecast alphas)")
print(f"w0 (unadj): {w0:.4f}")
print(f"wA (active): {wA:.4f}")
print(f"wM (passive): {wM:.4f}")
print(f"Overall beta: {wA * beta_A + wM * 1:.2f}")


TB Optimal Mix (Eq 8.24-25, forecast alphas)
w0 (unadj): 19.8983
wA (active): 0.9669
wM (passive): 0.0331
Overall beta: 0.05


In [14]:
# =============================================================================
# 8. PURE TB WEIGHTS (Active Securities Only)
# No explicit benchmark holding—wM implicit.
# =============================================================================
tb_weights = {t: wA * wAi[t] for t in securities}

print("\nPure TB Weights (active only, sum=wA)")
for t in sorted(tb_weights, key=lambda k: abs(tb_weights[k]), reverse=True)[:10]:
    print(f"{t:15} {tb_weights[t]*100:.2f}")


Pure TB Weights (active only, sum=wA)
IEF.O           19.88
VTIP.O          17.63
SDEU.L          14.52
5108.T          7.43
MSFT.O          6.66
IBGL.L          4.86
BLK.N           4.85
SASY.PA         4.13
BBRM.NS         3.51
AM.PA           2.39


In [15]:
# =============================================================================
# 9. CONSTRAINTS (ICM340 Fund Overrides)
# =============================================================================
max_pos_pct = 0.25
max_gearing = 1.50

final_weights = tb_weights.copy()

for t in final_weights:
    final_weights[t] = np.clip(final_weights[t], -max_pos_pct, max_pos_pct)

total_w = sum(final_weights.values())
grossexp = sum(abs(w) for w in final_weights.values())
while grossexp > max_gearing * abs(total_w) and abs(total_w) > 1e-6:
    scale = 0.95
    for t in final_weights:
        final_weights[t] *= scale
    total_w = sum(final_weights.values())
    grossexp = sum(abs(w) for w in final_weights.values())

if abs(total_w) > 1e-6:
    for t in final_weights:
        final_weights[t] /= total_w



totallong = sum(max(0, v) for v in final_weights.values())
totalshort = sum(-min(0, v) for v in final_weights.values())
grossexp = totallong + abs(totalshort)

print(f"\nConstrained (sum=100%, active only)")
print(f"Gearing gross/net {grossexp:.1f} / 100.0  <=150")
print(f"Max position {max(abs(v) for v in final_weights.values())*100:.1f} <=25")
print("\n" + "-"*50)
print("Asset".ljust(20) + "Weight".ljust(7) + "Value £50m".rjust(15))
print("-"*50)
for t in sorted(final_weights, key=lambda x: abs(final_weights[x]), reverse=True)[:20]:
    w = final_weights[t]
    print(f"{t}".ljust(20) + f"{w*100:6.1f}" + f"{w * portfolio_value:15,.0f}")


Constrained (sum=100%, active only)
Gearing gross/net 1.1 / 100.0  <=150
Max position 21.8 <=25

--------------------------------------------------
Asset               Weight      Value £50m
--------------------------------------------------
IEF.O                 21.8     10,917,648
VTIP.O                19.4      9,685,511
SDEU.L                15.9      7,974,445
5108.T                 8.2      4,082,193
MSFT.O                 7.3      3,656,150
IBGL.L                 5.3      2,667,951
BLK.N                  5.3      2,665,048
SASY.PA                4.5      2,266,548
BBRM.NS                3.9      1,926,968
AM.PA                  2.6      1,311,822
UTOS.SI               -2.5     -1,262,666
000660.KS              2.5      1,251,748
UBER.N                 1.9        963,242
6723.T                 1.6        817,763
CVX.N                  1.3        674,811
BABA.N                 0.8        387,316
FUTU.OQ                0.6        307,256
WISEa.L               -0.6       -293,754
M

In [16]:
# =============================================================================
# 10. VALIDATION
# =============================================================================
final_beta = sum(final_weights[t] * results[t]['beta'] for t in securities)
final_te_pct = np.sqrt(sum((final_weights[t]**2 * (results[t]['sigmae_annual_pct']/100)**2) for t in securities)) * 100
print(f"\nFinal beta {final_beta:.2f}")
print(f"Final TE (nonsystematic) {final_te_pct:.1f}%")


Final beta 0.02
Final TE (nonsystematic) 4.4%
